# Pretrained models

Selected model: https://huggingface.co/google/embeddinggemma-300m
- open embedding model from Google
- maximum input context length of 2048 tokens
- trained on web text, technical documents and common LLM-task instructions

While our main analyses uses the semantic distances generated by EmbeddingGemma300m, we validated the calculations with mpnet-base 
https://huggingface.co/sentence-transformers/all-mpnet-base-v2. Good consistency was demonstrated.

# Set up

In [ ]:
from sentence_transformers import SentenceTransformer
from sentence_transformers import util 
from itables import show
import pandas as pd
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

## Global variables

In [ ]:
# look i know im not supposed to hardcode the token but i dont wanna deal with setting up env vars rn
pretrained_model = SentenceTransformer("google/embeddinggemma-300m", token = "token")

# Basic data wrangling

## Chatlogs

In [ ]:
# import the dataset
root_dir = Path("/rootdir")
file_path = root_dir / "02_processed_data" / "output" / "cleaned_selected_dataset_no_artefacts.pkl"
tgaia = pd.read_pickle(file_path)
tgaia = tgaia.sort_values(["studId", "timestamp"], ascending=[True, True], kind="mergesort")
tgaia = tgaia.drop(columns=["systemPrompt", "chatId", "spId", "createBy", "language", "detected.emoji"])
tgaia = tgaia.reset_index(drop=True)

In [ ]:
tgaia['precd_response'] = tgaia.groupby('studId')['response'].shift(1)
tgaia['precd_prompt'] = tgaia.groupby('studId')['prompt'].shift(1)
show(tgaia.head(10))

In [ ]:
tgaia['dataset'].unique()

# Create the topic list

In [ ]:
tgaia_topics_dict = {
    'Vietnam_War': 'Historical reasons why the communists won the Vietnam War. The broad reasons are the strengths of the Viet Cong, the weaknesses of the US, the overdependence of the Southern government on the US, the support from Soviet Union and China and the rising anti-war movements.',
    'Self_Governance_SG': 'The significance of historical events leading to Singapore’s internal self-government. Historical significance is the extent of how events left a lasting impact, led to change, was memorable or affected many people',
    'Singapore_Significance': 'The significance of historical events leading to Singapore’s internal self-government. Historical significance is the extent of how events left a lasting impact, led to change, was memorable or affected many people',
    'Inclusive_Housing': 'Essay writing on to what extent does providing inclusive housing foster a strong sense of belonging among residents. The terms to be unpacked are "sense of belonging" and "provision of inclusive housing".',
    'Contemporary_SG': 'Social studies essay on the contemporary societal issues, and various social segments, in Singapore. ',
    'History_Teacher': 'Brief historical facts and how the factors were influenced by historical context, such as the implications of actions and events.',
    'AI_Aware': 'Different perspectives on whether artificial intelligence is helpful or harmful in different aspects of people’s lives.',
    'Knowing_Korea': 'Social studies knowledge for Korean history and culture.',
    'Unpacking_Poems': 'Read unseen literature and unpack the effective use of diction in a poem. Describe how diction is effectively used in the poem to create meaning as well as identify the key themes and ideas in a poem.'   
}

tgaia_topics = pd.DataFrame(list(tgaia_topics_dict.items()), columns=['dataset', 'task'])

## checks 
# show(tgaia_topics)
# print(len(tgaia_topics['dataset']))

# Compute prompt-prompt semantic similarity
- Dot product of normalised sentence embeddings gives us the cosine similarity

## Prompt -> Prompt

In [ ]:
# embeddings for prompts with history (i.e., those with preceding prompts)
tgaia_prompts_list_with_history = tgaia.loc[mask_no_openers, 'prompt'].tolist()
embeddings_prompts_with_history = pretrained_model.encode(tgaia_prompts_list_with_history, convert_to_tensor = True, normalize_embeddings = True)

# embeddings for preceding prompts
tgaia_prec_prompts_list = tgaia.loc[mask_no_openers, 'precd_prompt'].tolist()
embeddings_precdPrompts = pretrained_model.encode(tgaia_prec_prompts_list, convert_to_tensor = True, normalize_embeddings = True)

In [ ]:
p2p_pairwise_similarity = (embeddings_prompts_with_history * embeddings_precdPrompts).sum(dim=1).cpu().numpy()
print(p2p_pairwise_similarity) 
print(len(p2p_pairwise_similarity))

In [ ]:
# create a series with specific index, so it can join with the original dataframe accordingly; and avoid length mismatch error
p2p_pairwise_similarity_series = pd.Series(p2p_pairwise_similarity,
    index=tgaia.index[mask_no_openers],
    name='p2p')

tgaia = tgaia.join(p2p_pairwise_similarity_series)

In [ ]:
show(tgaia[['prompt', 'precd_prompt', 'precd_response', 'p2p']])

# Topic -> Prompt

In [ ]:
# define a function to compute semantic similarity for any given dataset and its corresponding topic
def compute_semantic_similarity(tgaia, tgaia_topics_dict):
    """
    compute cosine similarity between prompts and their corresponding topics.
    
    parameters:
    -----------
    tgaia : pd.DataFrame
        DataFrame with columns: studId, dataset, prompt
    tgaia_topics_dict : dict
        Dictionary mapping dataset -> topic string
    
    returns:
    --------
    pd.DataFrame
        original dataframe with added 'semantic_similarity' column
    """
    
    results = []
    
    # process each unique dataset
    for dataset in tgaia['dataset'].unique():
        # subset dataframe for current dataset
        subset_df = tgaia[tgaia['dataset'] == dataset].copy()
        
        # retrieve the corresponding topic
        topic = tgaia_topics_dict.get(dataset)
        
        if topic is None:
            print(f"Warning: No topic found for dataset '{dataset}'")
            subset_df['semantic_similarity'] = np.nan
            results.append(subset_df)
            continue
        
        # generate embeddings
        prompts_list = subset_df['prompt'].tolist()
        
        # embed the topic (single string)
        topic_embedding = pretrained_model.encode([topic]) 
        
        # embed all prompts
        prompts_embeddings = pretrained_model.encode(prompts_list)  
        
        # compute cosine similarity between each prompt and the topic
        similarities = cosine_similarity(prompts_embeddings, topic_embedding)
        subset_df['sys'] = similarities.flatten()
        
        results.append(subset_df)
    
    # concatenate all subsets back together
    final_df = pd.concat(results, ignore_index=True)
    
    return final_df

In [ ]:
tgaia_sys = compute_semantic_similarity(tgaia, tgaia_topics_dict)
show(tgaia_sys)

# Combine and output

In [ ]:
tgaia_sys.to_excel('semantic_dist.xlsx', index=False)